# Notebook 51 — Boundary of Universality and Failure Modes

This notebook characterizes where the metadata-conditioned meta-law starts to fail, which regime directions
drive that failure, and whether minimal refinements repair the hardest cases.

We keep the baseline sparse template:

\[
g(r,c;\beta)=\beta_0+\beta_1 c+\beta_2 r+\beta_3 c^3+\beta_4 r^2+\beta_5 r c^2
\]

and analyze the boundary between:
- strong interpolation
- mild extrapolation
- structured failure

## Main goals

1. Define distance to training support in regime space.
2. Measure error growth vs regime distance.
3. Diagnose failures via coefficient bias and residual fields.
4. Test whether one extra basis term repairs harder cases.
5. Compare global vs local meta-laws near the boundary.

In [ ]:
# Core imports

import warnings
warnings.filterwarnings("ignore")

import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor

try:
    from IPython.display import display
except Exception:
    display = print

np.random.seed(42)

In [ ]:
# Data discovery and synthetic fallback

DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = [
        "*residual*flow*.parquet",
        "*residual*flow*.csv",
        "*governing*flow*.parquet",
        "*governing*flow*.csv",
        "*.parquet",
        "*.csv",
        "*.pkl",
        "*.pickle",
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat))
        candidates.extend(glob.glob(os.path.join("data", pat)))
        candidates.extend(glob.glob(os.path.join("outputs", pat)))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        n_paths = 14
                        n_steps = 42
                        c_grid = np.linspace(-1.25, 1.05, n_steps)

                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.05, 5: 0.02, 7: 0.06}[k]
                        flow_shift = 0.05 if flow_mode == "nonlinear" else -0.02
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(n_paths):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift + flow_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if task == "zeta_vs_poisson":
                                    g -= 0.015 * c
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                delta_c = c_grid[min(window_id + 1, n_steps - 1)] - c if window_id < n_steps - 1 else c - c_grid[max(window_id - 1, 0)]
                                noise = 0.012 * np.random.randn()
                                next_r = r + (g + noise) * delta_c

                                rows.append(
                                    {
                                        "system": system,
                                        "task": task,
                                        "forcing_mode": forcing_mode,
                                        "k": k,
                                        "flow_mode": flow_mode,
                                        "condition_coord": c,
                                        "residual": r,
                                        "predicted_flow": g + noise,
                                        "next_residual": next_r,
                                        "delta_condition": delta_c,
                                        "sample_id": sample_id,
                                        "path_id": path_id,
                                        "window_id": window_id,
                                    }
                                )
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None
print("Selected DATA_PATH:", DATA_PATH)
print("USE_SYNTHETIC_FALLBACK:", USE_SYNTHETIC_FALLBACK)

In [ ]:
# Loading + schema alignment

def load_dataframe(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

alias_groups = {
    "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
    "residual": ["residual", "r", "resid"],
    "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
    "next_residual": ["next_residual", "r_next", "next_r"],
    "delta_condition": ["delta_condition", "dc", "delta_c"],
    "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
}

def align_schema(df):
    cols = list(df.columns)
    rename_map = {}
    for canonical, aliases in alias_groups.items():
        for a in aliases:
            if a in cols and a != canonical:
                rename_map[a] = canonical
                break
    df = df.rename(columns=rename_map)

    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]

    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        dc = np.gradient(df["condition_coord"].to_numpy())
        df["delta_condition"] = dc

    defaults = {
        "system": "unknown_system",
        "task": "unknown_task",
        "forcing_mode": "unknown_forcing",
        "k": 5,
        "flow_mode": "unknown_flow",
        "sample_id": np.arange(len(df)),
        "path_id": 0,
        "window_id": np.arange(len(df)),
    }
    for k, v in defaults.items():
        if k not in df.columns:
            df[k] = v

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")
    return df

if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)
print("Data source:", data_source)
print("Shape:", df.shape)
display(df.head())

In [ ]:
# Baseline template and regime datasets

TERM_NAMES = ["1", "c", "r", "c^3", "r^2", "r c^2"]
COEF_COLS = [f"coef_{t}" for t in TERM_NAMES]

def design_template(data, extra_term=None):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    cols = [
        np.ones_like(r),
        c,
        r,
        c**3,
        r**2,
        r * c**2,
    ]
    term_names = TERM_NAMES.copy()

    if extra_term == "rc":
        cols.append(r * c)
        term_names.append("r c")
    elif extra_term == "c^2":
        cols.append(c**2)
        term_names.append("c^2")
    elif extra_term == "c^5":
        cols.append(c**5)
        term_names.append("c^5")
    elif extra_term == "r^3":
        cols.append(r**3)
        term_names.append("r^3")
    elif extra_term == "r^2 c":
        cols.append((r**2) * c)
        term_names.append("r^2 c")
    elif extra_term == "r c^3":
        cols.append(r * (c**3))
        term_names.append("r c^3")

    return np.column_stack(cols), term_names

def fit_template(sub, alpha=1e-6, extra_term=None):
    X, term_names = design_template(sub, extra_term=extra_term)
    y = sub["predicted_flow"].to_numpy(dtype=float)
    beta = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ y)
    pred = X @ beta
    stats = {"n": len(sub), "rmse": math.sqrt(mean_squared_error(y, pred))}
    for name, coef in zip(term_names, beta):
        stats[f"coef_{name}"] = float(coef)
    return beta, pred, stats, term_names

def predict_with_beta(sub, beta, extra_term=None):
    X, _ = design_template(sub, extra_term=extra_term)
    return X @ beta

def safe_corr(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def explained_variance_score_manual(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.var(y_true)
    if denom < 1e-12:
        return np.nan
    return float(1.0 - np.var(y_true - y_pred) / denom)

def trajectory_gap(sub, beta_ref, beta_cmp, n_r0=15, extra_term=None):
    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 40)
    r0s = np.linspace(np.quantile(sub["residual"], 0.05), np.quantile(sub["residual"], 0.95), n_r0)
    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    def integrate(beta, r0):
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for i in range(len(cgrid) - 1):
            c = cgrid[i]
            dc = float(cgrid[i+1] - cgrid[i])
            x, _ = design_template(pd.DataFrame({"residual":[r], "condition_coord":[c]}), extra_term=extra_term)
            g = float(np.clip(x[0] @ beta, -flow_cap, flow_cap))
            r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        return np.array(traj)

    errs = []
    for r0 in r0s:
        t_ref = integrate(beta_ref, r0)
        t_cmp = integrate(beta_cmp, r0)
        errs.append(math.sqrt(mean_squared_error(t_ref, t_cmp)))
    return float(np.mean(errs))

In [ ]:
# Per-regime coefficient dataset

regime_rows = []
regime_subsets = {}
group_cols = ["system", "task", "forcing_mode", "k", "flow_mode"]

for vals, sub in df.groupby(group_cols):
    if len(sub) < 30:
        continue
    beta, pred, stats, term_names = fit_template(sub.copy(), extra_term=None)
    kval = int(vals[3]) if float(vals[3]).is_integer() else vals[3]
    regime_id = f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"
    row = {
        "regime_id": regime_id,
        "system": vals[0],
        "task": vals[1],
        "forcing_mode": vals[2],
        "k": float(vals[3]),
        "flow_mode": vals[4],
    }
    row.update(stats)
    regime_rows.append(row)
    regime_subsets[regime_id] = sub.copy()

coef_df = pd.DataFrame(regime_rows).reset_index(drop=True)
display(coef_df.head())

In [ ]:
# Metadata matrix

meta_df = coef_df[["regime_id", "system", "task", "forcing_mode", "flow_mode", "k"]].copy()
X_base = pd.get_dummies(meta_df[["system", "task", "forcing_mode", "flow_mode"]], drop_first=False)
X_base["k"] = coef_df["k"].astype(float).values
X_base["k2"] = coef_df["k"].astype(float).values**2
Y_coef = coef_df[COEF_COLS].copy()

display(X_base.head())
display(Y_coef.head())

In [ ]:
# Meta-model builders

class MultiTargetMLP:
    def __init__(self, hidden_layer_sizes=(16,), random_state=42, max_iter=4000):
        self.hidden_layer_sizes = hidden_layer_sizes
        self.random_state = random_state
        self.max_iter = max_iter
        self.models = None

    def fit(self, X, Y):
        Y = np.asarray(Y, dtype=float)
        if Y.ndim == 1:
            Y = Y[:, None]
        self.models = [
            MLPRegressor(hidden_layer_sizes=self.hidden_layer_sizes, random_state=self.random_state, max_iter=self.max_iter)
            for _ in range(Y.shape[1])
        ]
        for j, m in enumerate(self.models):
            m.fit(X, Y[:, j])
        return self

    def predict(self, X):
        cols = [m.predict(X) for m in self.models]
        return np.column_stack(cols)

def build_linear():
    return LinearRegression()

def build_ridge():
    return Ridge(alpha=1.0)

def build_mlp():
    return MultiTargetMLP()

def fit_predict_model(X_train, Y_train, X_test, model_name):
    builders = {
        "linear": build_linear,
        "ridge": build_ridge,
        "mlp_small": build_mlp,
    }
    return builders[model_name]().fit(np.asarray(X_train, float), np.asarray(Y_train, float)).predict(np.asarray(X_test, float))

def fit_predict_latent(X_train, Y_train, X_test, model_name):
    coef_scaler = StandardScaler()
    Y_train_std = coef_scaler.fit_transform(np.asarray(Y_train, float))
    pca = PCA(n_components=2, random_state=42)
    Z_train = pca.fit_transform(Y_train_std)
    zhat = fit_predict_model(X_train, Z_train, X_test, model_name)
    yhat_std = pca.inverse_transform(np.atleast_2d(zhat))
    yhat = coef_scaler.inverse_transform(yhat_std)
    return yhat

## Regime distance metrics

In [ ]:
# Distance from a held-out regime to training support

def regime_distance(row_a, row_b, w_k=1.0, w_system=1.0, w_task=1.0, w_forcing=1.0, w_flow=1.0):
    d = 0.0
    d += w_k * abs(float(row_a["k"]) - float(row_b["k"])) / 2.0
    d += w_system * float(row_a["system"] != row_b["system"])
    d += w_task * float(row_a["task"] != row_b["task"])
    d += w_forcing * float(row_a["forcing_mode"] != row_b["forcing_mode"])
    d += w_flow * float(row_a["flow_mode"] != row_b["flow_mode"])
    return d

def distance_to_support(test_row, train_df):
    ds = [regime_distance(test_row, tr) for _, tr in train_df.iterrows()]
    return min(ds), float(np.mean(ds))

# Example distances
example_rows = []
for i in range(min(5, len(coef_df))):
    test_row = coef_df.iloc[i]
    train_df = coef_df.drop(index=i)
    dmin, dmean = distance_to_support(test_row, train_df)
    example_rows.append({"regime_id": test_row["regime_id"], "min_dist": dmin, "mean_dist": dmean})
display(pd.DataFrame(example_rows))

## Use Notebook 50-style harder blocks as boundary probes

In [ ]:
# Define evaluation blocks

def make_block_masks():
    masks = {}

    # k tests
    masks["k_extrapolate_high"] = coef_df["k"].astype(float) == 7.0
    masks["k_extrapolate_low"] = coef_df["k"].astype(float) == 3.0
    masks["k_interpolate_mid"] = coef_df["k"].astype(float) == 5.0

    # families
    for fmode in sorted(coef_df["forcing_mode"].astype(str).unique()):
        masks[f"forcing_holdout::{fmode}"] = coef_df["forcing_mode"].astype(str) == fmode

    for system in sorted(coef_df["system"].astype(str).unique()):
        masks[f"system_holdout::{system}"] = coef_df["system"].astype(str) == system

    for task in sorted(coef_df["task"].astype(str).unique()):
        masks[f"task_holdout::{task}"] = coef_df["task"].astype(str) == task

    return masks

block_masks = make_block_masks()
print(list(block_masks.keys())[:10], "...")

In [ ]:
# Evaluate blocks and attach distance metrics

beta_shared = Y_coef.to_numpy(dtype=float).mean(axis=0)

def evaluate_block(test_mask, split_name):
    train_mask = ~test_mask

    X_train = X_base.loc[train_mask]
    X_test = X_base.loc[test_mask]
    Y_train = Y_coef.loc[train_mask]
    Y_test = Y_coef.loc[test_mask]
    train_df = coef_df.loc[train_mask].reset_index(drop=True)
    test_df = coef_df.loc[test_mask].reset_index(drop=True)

    rows = []
    for i in range(len(test_df)):
        regime_id = test_df.iloc[i]["regime_id"]
        beta_true = Y_test.iloc[i].to_numpy(dtype=float)
        x_te = X_test.iloc[[i]]

        candidates = {
            "linear": fit_predict_model(X_train, Y_train, x_te, "linear")[0],
            "ridge": fit_predict_model(X_train, Y_train, x_te, "ridge")[0],
            "mlp_small": fit_predict_model(X_train, Y_train, x_te, "mlp_small")[0],
            "latent_linear": fit_predict_latent(X_train, Y_train, x_te, "linear")[0],
            "latent_ridge": fit_predict_latent(X_train, Y_train, x_te, "ridge")[0],
            "latent_mlp_small": fit_predict_latent(X_train, Y_train, x_te, "mlp_small")[0],
        }

        best_name = min(candidates, key=lambda k: math.sqrt(mean_squared_error(beta_true, candidates[k])))
        beta_meta = candidates[best_name]

        sub = regime_subsets[regime_id]
        y_emp = sub["predicted_flow"].to_numpy(dtype=float)
        pred_shared = predict_with_beta(sub, beta_shared)
        pred_meta = predict_with_beta(sub, beta_meta)
        pred_specific = predict_with_beta(sub, beta_true)

        dmin, dmean = distance_to_support(test_df.iloc[i], train_df)

        rows.append({
            "split_name": split_name,
            "regime_id": regime_id,
            "best_meta_model": best_name,
            "min_support_dist": dmin,
            "mean_support_dist": dmean,

            "coef_rmse_meta": math.sqrt(mean_squared_error(beta_true, beta_meta)),
            "coef_rmse_shared": math.sqrt(mean_squared_error(beta_true, beta_shared)),

            "field_rmse_meta": math.sqrt(mean_squared_error(y_emp, pred_meta)),
            "field_rmse_shared": math.sqrt(mean_squared_error(y_emp, pred_shared)),
            "field_rmse_specific": math.sqrt(mean_squared_error(y_emp, pred_specific)),

            "field_corr_meta": safe_corr(y_emp, pred_meta),
            "field_corr_shared": safe_corr(y_emp, pred_shared),
            "field_corr_specific": safe_corr(y_emp, pred_specific),

            "traj_rmse_meta": trajectory_gap(sub, beta_true, beta_meta),
            "traj_rmse_shared": trajectory_gap(sub, beta_true, beta_shared),
        })
    return pd.DataFrame(rows)

block_results = []
for name, mask in block_masks.items():
    if mask.sum() == 0 or mask.sum() == len(coef_df):
        continue
    block_results.append(evaluate_block(mask, name))

boundary_df = pd.concat(block_results, ignore_index=True)
display(boundary_df.head())

## Error vs distance

In [ ]:
# Error vs support distance

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(boundary_df["min_support_dist"], boundary_df["coef_rmse_meta"])
axes[0].set_xlabel("min support distance")
axes[0].set_ylabel("coefficient RMSE")
axes[0].set_title("Coefficient error vs distance")

axes[1].scatter(boundary_df["min_support_dist"], boundary_df["field_rmse_meta"])
axes[1].set_xlabel("min support distance")
axes[1].set_ylabel("field RMSE")
axes[1].set_title("Field error vs distance")

axes[2].scatter(boundary_df["min_support_dist"], boundary_df["traj_rmse_meta"])
axes[2].set_xlabel("min support distance")
axes[2].set_ylabel("trajectory RMSE")
axes[2].set_title("Trajectory error vs distance")

plt.tight_layout()
plt.show()

In [ ]:
# Summary by block

boundary_summary = boundary_df.groupby("split_name")[[
    "min_support_dist", "coef_rmse_meta", "field_rmse_meta", "traj_rmse_meta"
]].mean().reset_index()

display(boundary_summary.sort_values("traj_rmse_meta", ascending=False))

## Failure decomposition and coefficient bias

In [ ]:
# Recover coefficient bias by term

coef_bias_rows = []
for _, row in boundary_df.iterrows():
    regime_id = row["regime_id"]

    # reconstruct best meta prediction again
    test_mask = coef_df["regime_id"] == regime_id
    train_mask = ~test_mask
    X_train = X_base.loc[train_mask]
    X_test = X_base.loc[test_mask]
    Y_train = Y_coef.loc[train_mask]
    Y_test = Y_coef.loc[test_mask]
    beta_true = Y_test.to_numpy(dtype=float)[0]

    candidates = {
        "linear": fit_predict_model(X_train, Y_train, X_test, "linear")[0],
        "ridge": fit_predict_model(X_train, Y_train, X_test, "ridge")[0],
        "mlp_small": fit_predict_model(X_train, Y_train, X_test, "mlp_small")[0],
        "latent_linear": fit_predict_latent(X_train, Y_train, X_test, "linear")[0],
        "latent_ridge": fit_predict_latent(X_train, Y_train, X_test, "ridge")[0],
        "latent_mlp_small": fit_predict_latent(X_train, Y_train, X_test, "mlp_small")[0],
    }
    best_name = min(candidates, key=lambda k: math.sqrt(mean_squared_error(beta_true, candidates[k])))
    beta_meta = candidates[best_name]

    for j, col in enumerate(COEF_COLS):
        coef_bias_rows.append({
            "split_name": row["split_name"],
            "regime_id": regime_id,
            "term": col,
            "bias": beta_true[j] - beta_meta[j],
            "abs_bias": abs(beta_true[j] - beta_meta[j]),
        })

coef_bias_df = pd.DataFrame(coef_bias_rows)
display(coef_bias_df.head())

In [ ]:
# Coefficient bias heatmap by block

bias_heat = coef_bias_df.groupby(["split_name", "term"])["abs_bias"].mean().reset_index().pivot(index="split_name", columns="term", values="abs_bias").fillna(0.0)

plt.figure(figsize=(10, 6))
plt.imshow(bias_heat.values, aspect="auto", cmap="magma")
plt.yticks(range(len(bias_heat.index)), bias_heat.index)
plt.xticks(range(len(bias_heat.columns)), bias_heat.columns, rotation=45, ha="right")
plt.colorbar(label="mean absolute coefficient bias")
plt.title("Coefficient bias by held-out block")
plt.tight_layout()
plt.show()

## Residual-field diagnostics in harder cases

In [ ]:
# Identify one easy and one hard case

easy_case = boundary_df.sort_values("traj_rmse_meta").iloc[0]
hard_case = boundary_df.sort_values("traj_rmse_meta", ascending=False).iloc[0]

display(easy_case.to_frame().T)
display(hard_case.to_frame().T)

In [ ]:
# Plot residual field diagnostics for easy/hard cases

def reconstruct_best_meta_for_regime(regime_id):
    test_mask = coef_df["regime_id"] == regime_id
    train_mask = ~test_mask
    X_train = X_base.loc[train_mask]
    X_test = X_base.loc[test_mask]
    Y_train = Y_coef.loc[train_mask]
    Y_test = Y_coef.loc[test_mask]
    beta_true = Y_test.to_numpy(dtype=float)[0]

    candidates = {
        "linear": fit_predict_model(X_train, Y_train, X_test, "linear")[0],
        "ridge": fit_predict_model(X_train, Y_train, X_test, "ridge")[0],
        "mlp_small": fit_predict_model(X_train, Y_train, X_test, "mlp_small")[0],
        "latent_linear": fit_predict_latent(X_train, Y_train, X_test, "linear")[0],
        "latent_ridge": fit_predict_latent(X_train, Y_train, X_test, "ridge")[0],
        "latent_mlp_small": fit_predict_latent(X_train, Y_train, X_test, "mlp_small")[0],
    }
    best_name = min(candidates, key=lambda k: math.sqrt(mean_squared_error(beta_true, candidates[k])))
    return beta_true, candidates[best_name], best_name

def plot_field_diff(regime_id, title_prefix):
    sub = regime_subsets[regime_id]
    beta_true, beta_meta, best_name = reconstruct_best_meta_for_regime(regime_id)

    subp = sub[["condition_coord", "residual", "predicted_flow"]].dropna().copy()
    subp["c_bin"] = pd.cut(subp["condition_coord"], bins=24, labels=False, include_lowest=True)
    subp["r_bin"] = pd.cut(subp["residual"], bins=24, labels=False, include_lowest=True)

    rows = []
    for (cb, rb), grp in subp.groupby(["c_bin", "r_bin"]):
        rows.append({
            "c_bin": cb,
            "r_bin": rb,
            "emp": grp["predicted_flow"].mean(),
            "specific": predict_with_beta(grp, beta_true).mean(),
            "meta": predict_with_beta(grp, beta_meta).mean(),
        })
    gdf = pd.DataFrame(rows)
    emp = gdf.pivot(index="r_bin", columns="c_bin", values="emp").sort_index().values
    meta = gdf.pivot(index="r_bin", columns="c_bin", values="meta").sort_index().values
    diff = meta - emp

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    ims = [
        axes[0].imshow(emp, aspect="auto", origin="lower", cmap="viridis"),
        axes[1].imshow(meta, aspect="auto", origin="lower", cmap="viridis"),
        axes[2].imshow(diff, aspect="auto", origin="lower", cmap="coolwarm"),
    ]
    axes[0].set_title("Empirical field")
    axes[1].set_title(f"Meta-law field ({best_name})")
    axes[2].set_title("Difference (meta - empirical)")
    fig.suptitle(f"{title_prefix}: {regime_id}", y=1.03)
    for ax, im in zip(axes, ims):
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

plot_field_diff(easy_case["regime_id"], "Easy case")
plot_field_diff(hard_case["regime_id"], "Harder case")

## Failure map in regime space

In [ ]:
# PCA on coefficient vectors, color by trajectory RMSE

coef_matrix = coef_df[COEF_COLS].to_numpy(dtype=float)
coef_std = StandardScaler().fit_transform(coef_matrix)
pca = PCA(n_components=2, random_state=42)
Z = pca.fit_transform(coef_std)

coef_df["pc1"] = Z[:, 0]
coef_df["pc2"] = Z[:, 1]

traj_err_map = boundary_df.groupby("regime_id")["traj_rmse_meta"].mean().to_dict()
coef_df["traj_rmse_meta"] = coef_df["regime_id"].map(traj_err_map)

plt.figure(figsize=(8, 5))
sc = plt.scatter(coef_df["pc1"], coef_df["pc2"], c=coef_df["traj_rmse_meta"], cmap="viridis")
for _, r in coef_df.iterrows():
    plt.text(r["pc1"], r["pc2"], r["forcing_mode"], fontsize=7, alpha=0.6)
plt.colorbar(sc, label="trajectory RMSE")
plt.title("Failure map in coefficient manifold")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()

## Minimal template extension tests

Test whether one extra basis term repairs harder cases.

In [ ]:
# Compare baseline vs extended templates on harder blocks

extra_terms = [None, "rc", "c^2", "c^5", "r^3", "r^2 c", "r c^3"]
target_blocks = [
    "k_extrapolate_high",
    "k_extrapolate_low",
    "system_holdout::entropy",
    "system_holdout::unevenness",
]

ext_rows = []
for block in target_blocks:
    if block not in block_masks:
        continue
    test_mask = block_masks[block]
    train_mask = ~test_mask

    X_train = X_base.loc[train_mask]
    X_test = X_base.loc[test_mask]
    train_regimes = coef_df.loc[train_mask, "regime_id"].tolist()
    test_regimes = coef_df.loc[test_mask, "regime_id"].tolist()

    for extra in extra_terms:
        # refit coefficients under extended template
        train_coef_rows = []
        test_coef_rows = []
        for rid in train_regimes:
            beta, _, stats, term_names = fit_template(regime_subsets[rid], extra_term=extra)
            train_coef_rows.append([stats[f"coef_{t}"] for t in term_names])
        for rid in test_regimes:
            beta, _, stats, term_names = fit_template(regime_subsets[rid], extra_term=extra)
            test_coef_rows.append([stats[f"coef_{t}"] for t in term_names])

        Xtr = np.asarray(X_train, float)
        Xte = np.asarray(X_test, float)
        Ytr = np.asarray(train_coef_rows, float)
        Yte = np.asarray(test_coef_rows, float)

        # simple best-of linear/ridge for extension comparison
        preds = {
            "linear": fit_predict_model(Xtr, Ytr, Xte, "linear"),
            "ridge": fit_predict_model(Xtr, Ytr, Xte, "ridge"),
        }

        best_name = min(preds, key=lambda k: math.sqrt(mean_squared_error(Yte.ravel(), preds[k].ravel())))
        Yhat = preds[best_name]

        # average trajectory error under extended template
        traj_errs = []
        for i, rid in enumerate(test_regimes):
            beta_true = Yte[i]
            beta_meta = Yhat[i]
            traj_errs.append(trajectory_gap(regime_subsets[rid], beta_true, beta_meta, extra_term=extra))

        ext_rows.append({
            "block": block,
            "extra_term": "baseline" if extra is None else extra,
            "best_model": best_name,
            "coef_rmse": math.sqrt(mean_squared_error(Yte.ravel(), Yhat.ravel())),
            "traj_rmse": float(np.mean(traj_errs)),
        })

extension_df = pd.DataFrame(ext_rows)
display(extension_df.head(20))

In [ ]:
# Plot template extension gains

for block in extension_df["block"].unique():
    sub = extension_df[extension_df["block"] == block].sort_values("traj_rmse")
    plt.figure(figsize=(8, 4))
    plt.bar(sub["extra_term"], sub["traj_rmse"])
    plt.title(f"Template extension test — {block}")
    plt.ylabel("mean trajectory RMSE")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## Global vs local meta-law near the boundary

In [ ]:
# Compare global vs nearest-neighbor local linear model on harder cases

hard_blocks = [
    "system_holdout::entropy",
    "system_holdout::unevenness",
    "k_extrapolate_high",
    "k_extrapolate_low",
]

local_rows = []
for block in hard_blocks:
    if block not in block_masks:
        continue
    test_mask = block_masks[block]
    train_mask = ~test_mask

    X_train = X_base.loc[train_mask]
    Y_train = Y_coef.loc[train_mask]
    train_meta = coef_df.loc[train_mask].reset_index(drop=True)
    test_meta = coef_df.loc[test_mask].reset_index(drop=True)

    for i in range(len(test_meta)):
        test_row = test_meta.iloc[i]
        regime_id = test_row["regime_id"]
        beta_true = Y_coef.loc[coef_df["regime_id"] == regime_id].to_numpy(dtype=float)[0]

        # global
        beta_global = fit_predict_model(X_train, Y_train, X_base.loc[coef_df["regime_id"] == regime_id], "linear")[0]

        # local nearest neighbors in metadata distance
        train_meta_with_dist = train_meta.copy()
        train_meta_with_dist["dist"] = train_meta_with_dist.apply(lambda r: regime_distance(test_row, r), axis=1)
        nn = train_meta_with_dist.sort_values("dist").head(min(8, len(train_meta_with_dist)))
        X_nn = X_base.loc[coef_df["regime_id"].isin(nn["regime_id"])]
        Y_nn = Y_coef.loc[coef_df["regime_id"].isin(nn["regime_id"])]
        beta_local = fit_predict_model(X_nn, Y_nn, X_base.loc[coef_df["regime_id"] == regime_id], "linear")[0]

        sub = regime_subsets[regime_id]
        local_rows.append({
            "block": block,
            "regime_id": regime_id,
            "global_traj_rmse": trajectory_gap(sub, beta_true, beta_global),
            "local_traj_rmse": trajectory_gap(sub, beta_true, beta_local),
        })

local_df = pd.DataFrame(local_rows)
display(local_df.head())

In [ ]:
# Plot global vs local comparison

summary_local = local_df.groupby("block")[["global_traj_rmse", "local_traj_rmse"]].mean().reset_index()

x = np.arange(len(summary_local))
w = 0.38
plt.figure(figsize=(9, 4))
plt.bar(x - w/2, summary_local["global_traj_rmse"], width=w, label="global linear meta-law")
plt.bar(x + w/2, summary_local["local_traj_rmse"], width=w, label="local nearest-neighbor linear")
plt.xticks(x, summary_local["block"], rotation=30, ha="right")
plt.ylabel("mean trajectory RMSE")
plt.title("Global vs local meta-law on harder blocks")
plt.legend()
plt.tight_layout()
plt.show()

## Final verdicts

In [ ]:
# Block-level verdict table

def block_verdict(name):
    sub = boundary_df[boundary_df["split_name"] == name]
    if len(sub) == 0:
        return "not evaluated"
    if sub["traj_rmse_meta"].mean() < 0.01:
        return "universality strong"
    if sub["traj_rmse_meta"].mean() < 0.04:
        return "universality moderate"
    return "universality boundary visible"

verdict_rows = []
for name in boundary_df["split_name"].unique():
    verdict_rows.append({
        "block": name,
        "min_support_dist_mean": boundary_df.loc[boundary_df["split_name"] == name, "min_support_dist"].mean(),
        "coef_rmse_meta_mean": boundary_df.loc[boundary_df["split_name"] == name, "coef_rmse_meta"].mean(),
        "field_rmse_meta_mean": boundary_df.loc[boundary_df["split_name"] == name, "field_rmse_meta"].mean(),
        "traj_rmse_meta_mean": boundary_df.loc[boundary_df["split_name"] == name, "traj_rmse_meta"].mean(),
        "verdict": block_verdict(name),
    })

verdict_df = pd.DataFrame(verdict_rows).sort_values("traj_rmse_meta_mean", ascending=False)
display(verdict_df)

## Working conclusion

Notebook 51 turns generalization error into structure.

A strong result is:
- error grows systematically with regime distance
- coefficient bias concentrates in a few terms
- failures cluster in identifiable regions of regime space
- one small template extension or local meta-law reduces the hardest-case error

If that pattern holds on your real exports, the next notebook should be:

**Notebook 52 — piecewise meta-law and adaptive template selection**